In [1]:
import numpy as np
import plotly.graph_objects as go

from scipy.optimize import minimize

from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
X, y = make_moons(
    n_samples=150,
    noise=0.15,
    random_state=42
)


y = np.where(y == 0, -1, 1)

In [3]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=X[y == -1, 0],
        y=X[y == -1, 1],
        mode="markers",
        name="Class -1"
    )
)

fig.add_trace(
    go.Scatter(
        x=X[y == 1, 0],
        y=X[y == 1, 1],
        mode="markers",
        name="Class +1"
    )
)

fig.update_layout(
    title="Nonlinear Dataset — make_moons",
    xaxis_title="X₁",
    yaxis_title="X₂"
)

fig.show()

In [4]:
class RBFSVM:

    def __init__(self, C=10.0, gamma=2.0):

        self.C = C
        self.gamma = gamma

        self.alpha = None
        self.b = None

        self.support_indices_ = None
        self.support_vectors_ = None


    def rbf_kernel(self,X1, X2, gamma):

        squared_distances = (
            np.sum(X1**2, axis=1)[:, None]
            + np.sum(X2**2, axis=1)[None, :]
            - 2 * X1 @ X2.T
        )
        return np.exp(
            -gamma * squared_distances
        )

    def fit(self, X, y):

        n_samples = X.shape[0]
        K = self.rbf_kernel(
            X,
            X,
            self.gamma
        )

        Q = (
            y[:, None]
            * y[None, :]
            * K
        )
        def objective(alpha):

            return (
                0.5 * alpha @ Q @ alpha
                - np.sum(alpha)
            )

        def gradient(alpha):

            return Q @ alpha - 1

        constraints = {
            "type": "eq",
            "fun": lambda alpha: np.dot(alpha, y),
            "jac": lambda alpha: y
        }

        bounds = [
            (0, self.C)
            for _ in range(n_samples)
        ]

        alpha_initial = np.zeros(n_samples)

        result = minimize(
            objective,
            alpha_initial,
            jac=gradient,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={
                "ftol": 1e-9,
                "maxiter": 2000,
                "disp": False
            }
        )

        if not result.success:

            raise RuntimeError(
                result.message
            )


        self.alpha = result.x

        self.support_indices_ = np.where(
            self.alpha > 1e-5
        )[0]

        self.support_vectors_ = X[
            self.support_indices_
        ]

        free_sv = np.where(
            (self.alpha > 1e-5)
            &
            (self.alpha < self.C - 1e-5)
        )[0]


        if len(free_sv) > 0:

            b_values = []

            for i in free_sv:

                decision_without_b = np.sum(
                    self.alpha
                    * y
                    * K[:, i]
                )

                b_i = y[i] - decision_without_b

                b_values.append(b_i)

            self.b = np.mean(b_values)


        else:

            self.b = 0.0


        return self


    def decision_function(self, X):

        K = self.rbf_kernel(
            X,
            self.support_vectors_,
            self.gamma
        )

        alpha_sv = self.alpha[
            self.support_indices_
        ]

        y_sv = y[
            self.support_indices_
        ]

        return (
            K @ (alpha_sv * y_sv)
            + self.b
        )


    def predict(self, X):

        scores = self.decision_function(X)

        return np.where(
            scores >= 0,
            1,
            -1
        )

In [5]:
C = 10.0
gamma = 2.0

scratch_svm = RBFSVM(
    C=C,
    gamma=gamma
)

scratch_svm.fit(X, y)

print("Scratch RBF SVM")
print("----------------")

print("C:", C)
print("gamma:", gamma)

print(
    "Support vectors:",
    len(scratch_svm.support_indices_)
)

print(
    "b:",
    scratch_svm.b
)

Scratch RBF SVM
----------------
C: 10.0
gamma: 2.0
Support vectors: 16
b: 0.12160356930374024


In [6]:
sklearn_svm = SVC(
    kernel="rbf",
    C=C,
    gamma=gamma
)

sklearn_svm.fit(X, y)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10.0
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",2.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [7]:
scratch_pred = scratch_svm.predict(X)

sklearn_pred = sklearn_svm.predict(X)


print("\nComparison")
print("----------------")

print(
    "Scratch accuracy:",
    accuracy_score(y, scratch_pred)
)

print(
    "Sklearn accuracy:",
    accuracy_score(y, sklearn_pred)
)

print(
    "Scratch support vectors:",
    len(scratch_svm.support_indices_)
)

print(
    "Sklearn support vectors:",
    len(sklearn_svm.support_)
)


Comparison
----------------
Scratch accuracy: 0.9933333333333333
Sklearn accuracy: 0.9933333333333333
Scratch support vectors: 16
Sklearn support vectors: 16


In [8]:
x1 = np.linspace(
    X[:, 0].min() - 0.5,
    X[:, 0].max() + 0.5,
    200
)

x2 = np.linspace(
    X[:, 1].min() - 0.5,
    X[:, 1].max() + 0.5,
    200
)

xx, yy = np.meshgrid(x1, x2)

grid = np.c_[
    xx.ravel(),
    yy.ravel()
]

decision = scratch_svm.decision_function(grid)

decision = decision.reshape(
    xx.shape
)

In [9]:
fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=X[y == -1, 0],
        y=X[y == -1, 1],
        mode="markers",
        name="Class -1",
        marker=dict(size=7)
    )
)

fig.add_trace(
    go.Scatter(
        x=X[y == 1, 0],
        y=X[y == 1, 1],
        mode="markers",
        name="Class +1",
        marker=dict(size=7)
    )
)

sv = scratch_svm.support_indices_

fig.add_trace(
    go.Scatter(
        x=X[sv, 0],
        y=X[sv, 1],
        mode="markers",
        name="Support Vectors",
        marker=dict(
            size=13,
            symbol="diamond",
            line=dict(width=2)
        )
    )
)


fig.add_trace(
    go.Contour(
        x=x1,
        y=x2,
        z=decision,
        contours=dict(
            start=0,
            end=0,
            size=1,
            coloring="lines"
        ),
        showscale=False,
        name="Decision Boundary"
    )
)


fig.add_trace(
    go.Contour(
        x=x1,
        y=x2,
        z=decision,
        contours=dict(
            start=1,
            end=1,
            size=1,
            coloring="lines"
        ),
        line=dict(color="red", width=2, dash="dash"),
        showscale=False,
        name="+1 Margin"
    )
)


fig.add_trace(
    go.Contour(
        x=x1,
        y=x2,
        z=decision,
        contours=dict(
            start=-1,
            end=-1,
            size=1,
            coloring="lines"
        ),
        line=dict(color="blue", width=2, dash="dash"),
        showscale=False,
        name="-1 Margin"
    )
)


fig.update_layout(
    title="RBF Soft-Margin SVM",
    xaxis_title="X₁",
    yaxis_title="X₂",
    width=900,
    height=700
)

fig.show()